# Lab 4 — Build the Interactive Tayseer Dashboard
### SDA-DSC-112 · Module 4 · Interactive Dashboards and BI Tools · **100 minutes (4a + 4b)**

| | |
|---|---|
| **Objective** | Compose a multi-view interactive dashboard (KPI headline, national trend, regional breakdown) in Plotly, add cross-filtering, drill-down and a measure toggle in Dash, reproduce the measures in a BI tool, and usability-test the result |
| **Dataset** | `tayseer_services.csv` · all `dim_*` · `kpi_targets.csv` · `region_directors.csv` · `dashboard_usability_tests.csv` |
| **Output** | `out/lab4_dashboard.html` (self-contained) · `app/dashboard.py` (runnable Dash app) · `out/lab4_bi_measures.md` |

### Lab 4a — build *(50 min)*
| Task | Minutes |
|---|---|
| 1 · The dashboard brief | 5 |
| 2 · The inverted-pyramid layout in Plotly | 15 |
| 3 · `national_adoption()` — and prove the naive version is wrong | 10 |
| 4 · Target reference lines and a takeaway title | 10 |
| 5 · Export a self-contained HTML and check the default state | 10 |

### Lab 4b — interactivity *(50 min)*
| Task | Minutes |
|---|---|
| 6 · Dash: measure toggle + click-to-cross-filter | 15 |
| 7 · Make state legible, add a reset | 10 |
| 8 · Drill path national → region → service, with a breadcrumb | 10 |
| 9 · Reproduce the measures in Power BI / Tableau, with RLS | 10 |
| 10 · Usability test | 5 |

### Setup

A dashboard is not a chart gallery. It is **one question**, answered in its default
state, refined by interaction. Modules 1–3 do not stop applying because the medium
became interactive.

In [1]:
import sys, pathlib, subprocess, textwrap
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import tayseer_viz as tv
tv.apply_theme()

df = tv.load_facts()
kpis = tv.load("kpi_targets.csv")
directors = tv.load("region_directors.csv")
services = tv.load("dim_services.csv")

APP_DIR = pathlib.Path.cwd() / "app"
APP_DIR.mkdir(exist_ok=True)
print(f"{len(df):,} fact rows · {len(kpis)} KPIs · {len(directors)} region directors")

# Your figures go to out/starter/ so they never overwrite the reference set.
tv.set_out_dir(tv.OUT / "starter")

28,080 fact rows · 10 KPIs · 13 region directors


PosixPath('/home/claude/work/pkg/notebooks/out/starter')

---
## Task 1 — The dashboard brief *(5 min)*

One sentence, three parts: **audience**, **decision**, **cadence**. Everything that
follows is justified against it, and anything that is not is cut.

Write it before you open a plotting library. A dashboard without a brief becomes a wall
of gauges, which is the failure mode this whole module exists to prevent.

In [2]:
# --- TODO -----------------------------------------------------------------
# Write BRIEF: a dict with keys
#   audience, decision, cadence, primary_question,
#   answered_in_default_state (bool), interaction_is_for
# One sentence each. This is the contract every later task is judged against.
BRIEF = None
# --------------------------------------------------------------------------

if BRIEF is None:
    print("TODO not implemented yet — expected 6 keys.")
    print("Hint: the primary question for Tayseer is 'is the programme on track,")
    print("      and where is it not?'  Everything on the dashboard serves that.")
else:
    for k, v in BRIEF.items():
        print(f"{k:26} {v}")
    assert BRIEF["answered_in_default_state"] is True, "zero clicks, or it is not a dashboard"

TODO not implemented yet — expected 6 keys.
Hint: the primary question for Tayseer is 'is the programme on track,
      and where is it not?'  Everything on the dashboard serves that.


---
## Task 3 — `national_adoption()`, and proof that the naive version is wrong *(10 min)*

This is the highest-value ten minutes in the course. Implement the measure correctly and
**print both numbers**. Most participants have shipped the naive version in production.

In [3]:
# --- TODO -----------------------------------------------------------------
# 1. national_adoption(frame) -> Series indexed by month, the CORRECT KPI-01:
#       100 * SUM(adoption/100 * unique_users) / SUM(unique_users)
# 2. naive_adoption(frame)    -> Series, the bug: groupby(month)[col].mean()
# 3. Print both latest values and the error in percentage points.
# 4. Build `err`: the same two numbers under six different filters, to show the
#    error is NOT a constant offset — it moves as the filter moves.
national_adoption = naive_adoption = None
err = None
# --------------------------------------------------------------------------

if national_adoption is None:
    print("TODO not implemented yet — expected correct 63.8% vs naive 61.2% (error +2.64 pp),")
    print("  and an `err` table showing the gap changing under each filter.")
else:
    nat, nav = national_adoption(df), naive_adoption(df)
    print(f"correct {nat.iloc[-1]:.1f}%   naive {nav.iloc[-1]:.1f}%")
    assert round(nat.iloc[-1], 1) == 63.8 and round(nav.iloc[-1], 1) == 61.2
    print("ok")

TODO not implemented yet — expected correct 63.8% vs naive 61.2% (error +2.64 pp),
  and an `err` table showing the gap changing under each filter.


---
## Task 2 — The inverted-pyramid layout in Plotly *(15 min)*

Three bands, top to bottom, in decreasing generality:

1. **KPI headline** — the "am I on track?" glance. One number and its delta vs target.
2. **Primary view** — the national trend with the target as a reference line.
3. **Breakdown** — the sorted regional bar, laggards highlighted (the Module 1 pop).

The eye path runs top-left → down. Nothing below the fold is required to answer the
primary question.

In [4]:
TARGET_A, TARGET_C = tv.TARGET_ADOPTION, tv.TARGET_COST
latest = df[df.month == df.month.max()]

# --- TODO -----------------------------------------------------------------
# Write build_dashboard(df) -> (fig, nat, reg) using make_subplots with:
#   specs=[[{"type": "indicator"}, {"type": "xy"}],
#          [{"type": "xy", "colspan": 2}, None]]
#   (1) row 1 col 1 : go.Indicator(mode="number+delta", value=<national adoption>,
#                                  delta={"reference": 65})
#   (2) row 1 col 2 : go.Scatter national trend + a dashed reference line at 65
#   (3) row 2 col 1 : go.Bar(orientation="h") sorted, laggards in tv.ACCENT and the
#                     rest in tv.GREY, plus a dashed reference line at 65
#   GOTCHA: fig.add_hline(row=, col=) walks EVERY subplot, and the Indicator has no
#   axes, so it raises. Use fig.add_shape with explicit refs taken from the traces:
#       tx, ty = fig.data[1].xaxis or "x", fig.data[1].yaxis or "y"
#       fig.add_shape(type="line", xref=f"{tx} domain", yref=ty, x0=0, x1=1, y0=65, y1=65,
#                     line=dict(dash="dash", color=tv.ACCENT))
# Layout: template="plotly_white", showlegend=False, and a TAKEAWAY title.
build_dashboard = None
# --------------------------------------------------------------------------

if build_dashboard is None:
    print("TODO not implemented yet — expected a three-band figure:")
    print("  KPI 63.8% (delta -1.2 pp vs target) / trend with target line /")
    print("  13 sorted bars with 9 below target, 5 of them stalled.")
else:
    dash_fig, nat_series, reg_latest = build_dashboard(df)
    dash_fig.show()

TODO not implemented yet — expected a three-band figure:
  KPI 63.8% (delta -1.2 pp vs target) / trend with target line /
  13 sorted bars with 9 below target, 5 of them stalled.


---
## Task 4 — Reference lines and a takeaway title *(10 min)*

Both are already in `build_dashboard`. This task is to check them and to see what the
dashboard looks like without: a version with topic titles and no target lines, so the
class can see that the *same three charts* stop supporting a decision.

In [5]:
# --- TODO -----------------------------------------------------------------
# 1. Build `checks`: one row per requirement (KPI delta, trend hline, bar vline,
#    takeaway title, brief echoed, laggards popped, no gauges/donuts/dual axes)
#    with columns ['Check', 'How', 'Value', 'Pass'] and assert they all pass.
# 2. Then build the SAME three views with TOPIC titles and NO reference lines,
#    so the class can see the decision support disappear.
checks = None
# --------------------------------------------------------------------------

if checks is None:
    print("TODO not implemented yet — expected 7 checks, all passing.")
else:
    display(checks); assert checks.Pass.all()

TODO not implemented yet — expected 7 checks, all passing.


---
## Task 5 — Export a self-contained HTML and check the default state *(10 min)*

`write_html` with `include_plotlyjs="inline"` produces one file that opens anywhere with
no network. Open it, and answer the primary question **without clicking anything**.

In [6]:
# --- TODO -----------------------------------------------------------------
# 1. dash_fig.write_html(tv.OUT / "lab4_dashboard.html", include_plotlyjs="inline")
#    ("inline" = self-contained; "cdn" needs a network and will be blank offline)
# 2. Print the zero-click answers: on track? where is it not? how many stalled?
# 3. Build a static matplotlib mirror for the slide deck and save it.
html_path = None
# --------------------------------------------------------------------------

if html_path is None:
    print("TODO not implemented yet — expected a few-MB self-contained HTML, and")
    print("  63.8% (-1.2 pp), 9 regions below target, 5 of them stalled.")
else:
    print("wrote", html_path, f"({html_path.stat().st_size/1e6:.1f} MB)")

TODO not implemented yet — expected a few-MB self-contained HTML, and
  63.8% (-1.2 pp), 9 regions below target, 5 of them stalled.


---
# Lab 4b — Interactivity
## Task 6 — Dash: measure toggle + click-to-cross-filter *(15 min)*

A Dash app cannot run inside a notebook cell, so it lives in `app/dashboard.py` and we
write it from here. Run it with:

```bash
python app/dashboard.py     # -> http://127.0.0.1:8050
```

The four interaction patterns, and which the app uses:

| Pattern | User task | Here |
|---|---|---|
| **Filter** | "only show the app channel" | channel dropdown |
| **Cross-filter** | "click a region, update every view" | click a bar → trend re-scopes |
| **Drill** | "national → region → service" | breadcrumb, Task 8 |
| **Toggle** | "show me cost instead" | measure dropdown |

In [7]:
# --- TODO -----------------------------------------------------------------
# Write app/dashboard.py. It must contain, at minimum:
#   * MEASURES: {column: (label, weight_column, target, direction)} for adoption,
#     cost and CSAT — note the weight differs per measure (kpi_targets.csv)
#   * measure(frame, col) that RECOMPUTES from the weight column, never .mean()
#   * dcc.Dropdown(id="measure")            <- the TOGGLE pattern
#   * dcc.Graph(id="region-bar")            <- click to CROSS-FILTER
#   * a callback taking Input("region-bar", "clickData") that re-scopes the trend
#   * if __name__ == "__main__": app.run(debug=True, port=8050)
# Then compile() it and subprocess-import it — it must at least import cleanly.
APP_SOURCE = None
# --------------------------------------------------------------------------

if APP_SOURCE is None:
    print("TODO not implemented yet — expected app/dashboard.py to compile and import,")
    print("  exposing app.layout, >= 4 callbacks, and MEASURES with 3 entries.")
else:
    app_path = APP_DIR / "dashboard.py"
    app_path.write_text(APP_SOURCE, encoding="utf-8")
    compile(APP_SOURCE, str(app_path), "exec")
    print("syntax check: OK ->", app_path)

TODO not implemented yet — expected app/dashboard.py to compile and import,
  exposing app.layout, >= 4 callbacks, and MEASURES with 3 entries.


---
## Task 7 — Make state legible, add a reset *(10 min)*

The **filter-state fog** simulation: hide the active filter and testers reliably misread a
filtered view as the national picture. State legibility is a **correctness** feature, not
a nicety.

Prove it with numbers: the same tile, filtered and unfiltered, and the size of the misread.

In [8]:
latest = df[df.month == df.month.max()]

# --- TODO -----------------------------------------------------------------
# 1. Build `fog`: the headline measures under 5 different scopes, plus a column
#    showing how far a tester would be wrong if they misread the filtered view
#    as the national one.
# 2. Build `affordances`: >= 5 ways your app makes state legible, with columns
#    ['Affordance', 'Implementation', 'Why it is a correctness feature'].
fog = affordances = None
# --------------------------------------------------------------------------

if fog is None:
    print("TODO not implemented yet — expected national 63.8% vs 'Branch only' and")
    print("  'Jazan only' scopes that differ by more than 10 points.")
else:
    display(fog); display(affordances)

TODO not implemented yet — expected national 63.8% vs 'Branch only' and
  'Jazan only' scopes that differ by more than 10 points.


---
## Task 8 — Drill path with a breadcrumb *(10 min)*

`national → region → service category → channel`, with the crumb always visible.
The rule: **the measure must be recomputed at every level** — you cannot cache a
percentage from the level above and reuse it.

In [9]:
# --- TODO -----------------------------------------------------------------
# Write drill_path(df, region=None, category=None, col=...) returning
#   (breadcrumb_string, grouping_key, level_frame, value_at_this_level)
# where the grouping key is:
#   region -> when nothing is selected
#   service_category -> when a region is selected
#   channel -> when a region AND a category are selected
# The measure must be RECOMPUTED at each level (tv.by / tv.adoption), never inherited.
# Render the three steps side by side, and show the error you would make by
# averaging the level below instead of recomputing.
drill_path = None
# --------------------------------------------------------------------------

if drill_path is None:
    print("TODO not implemented yet — expected three panels:")
    print("  National (13 regions) > Jazan (9 categories) > Jazan/Health (4 channels)")
else:
    print(drill_path(df)[0], "|", drill_path(df, "Jazan")[0])

TODO not implemented yet — expected three panels:
  National (13 regions) > Jazan (9 categories) > Jazan/Health (4 channels)


---
## Task 9 — The same measures in Power BI / Tableau, with RLS *(10 min)*

The semantic layer is tool-agnostic first, tool-specific second. Write it out, then
demonstrate row-level security against `region_directors.csv`: a region director sees
one region, and the national number they see is *their* region's, correctly recomputed.

In [10]:
# --- TODO -----------------------------------------------------------------
# 1. Write BI_MEASURES (markdown): the additivity table, the DAX measures
#    (Digital Adoption %, Cost per Transaction, CSAT, Delta vs Target,
#    Regions Below Target), the Tableau equivalents, and the RLS role.
#    Save it to tv.OUT / "lab4_bi_measures.md".
# 2. Write rls_scope(email) -> (region, filtered_frame). Deny by default when the
#    email has no mapping in region_directors.csv.
# 3. Build `rls`: for five directors, what they see and the adoption % they see.
BI_MEASURES = rls_scope = rls = None
# --------------------------------------------------------------------------

if rls is None:
    print("TODO not implemented yet — expected each director to see exactly 1 region")
    print("  and 2,160 fact rows (60 months x 9 categories x 4 channels).")
else:
    display(rls); assert (rls["regions visible"] == 1).all()

TODO not implemented yet — expected each director to see exactly 1 region
  and 2,160 fact rows (60 months x 9 categories x 4 channels).


---
## Task 10 — Usability test *(5 min)*

`dashboard_usability_tests.csv` holds 144 timed task results: the same users, the same
questions, on the **v1 gauge wall** and the **v2 inverted pyramid**. This is the Module 4
benchmark, as data.

In [11]:
ut = tv.load("dashboard_usability_tests.csv")

# --- TODO -----------------------------------------------------------------
# 1. `summary`: by dashboard_version, the median/mean time_to_answer_sec,
#    the answer_correct rate, the gave_up rate, mean clicks and confidence.
# 2. Print the v1 -> v2 change on each, and assert v2 median < 15 s (the benchmark).
# 3. Chart it: per-task median times side by side, plus correct / gave-up rates.
# 4. Build the Module 4 `benchmark` comparison table.
summary = None
# --------------------------------------------------------------------------

if summary is None:
    print("TODO not implemented yet — expected roughly:")
    print("  v1 gauge wall      : median 72 s, 24% correct, 31% gave up, 6.7 clicks")
    print("  v2 inverted pyramid: median 11 s, 86% correct,  0% gave up, 1.9 clicks")
else:
    display(summary)

TODO not implemented yet — expected roughly:
  v1 gauge wall      : median 72 s, 24% correct, 31% gave up, 6.7 clicks
  v2 inverted pyramid: median 11 s, 86% correct,  0% gave up, 1.9 clicks


---
## Lab 4 checklist

**4a** — [ ] brief written · [ ] three-band layout · [ ] 63.8% vs 61.2% printed ·
[ ] reference lines + takeaway title · [ ] self-contained HTML, zero-click answer

**4b** — [ ] `app/dashboard.py` compiles and imports · [ ] measure toggle ·
[ ] click-to-cross-filter · [ ] state always visible + reset · [ ] drill with breadcrumb ·
[ ] DAX / Tableau measures + RLS · [ ] usability numbers reproduced

Save your work as `lab4_solution.ipynb`.

---

## Section 4b-alt — Build it both ways: animation versus small multiples  *(20 min)*

Robertson et al. (IEEE TVCG 2008) measured the same trade-off this section reproduces: for ANALYSIS,
small multiples were fastest (45.7 s) and most accurate; animation was slowest (83.1 s). For
PRESENTATION the order reverses. Build both views of the Tayseer adoption trend and find out for yourself.


### Task 1 · (6 min) The animated view

Build the animated regional view with a month slider, region colour and a play control.

In [12]:
import tayseer_viz as tv
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import plotly.express as px
tv.apply_theme()

df = tv.load_facts()
# TODO: build `mr` = month x region adoption (user-weighted — NEVER .mean()) plus transactions,
#       then a plotly animated scatter with animation_frame on the month.
mr = None
if mr is None:
    print("TODO not implemented yet — expected 780 rows (60 months x 13 regions)")

TODO not implemented yet — expected 780 rows (60 months x 13 regions)


### Task 2 · (5 min) The small-multiples view

Same 13 series as a grid, each panel drawn against the other twelve in grey, ordered by latest value.

In [13]:
# TODO: draw the 13 series as a 3x5 small-multiples grid, each panel against the
#       other twelve in grey, ordered by latest adoption, with the target line on every panel.
built = None
if built is None:
    print("TODO not implemented yet — expected a 3x5 grid with 13 used panels")

TODO not implemented yet — expected a 3x5 grid with 13 used panels


### Task 3 · (5 min) The comparison question, timed

Your partner asks, without warning: **among the six largest regions, which pair changed places between month 20 and month 44?**
Answer it from the animation, then from the grid. Time both. The question deliberately uses two
NON-ADJACENT time points — that is where animation fails.

In [14]:
# TODO: among the six largest regions by transactions, find every pair whose adoption
#       gap changes sign between month 20 and month 44. Time yourself on each view first.
crossings = None
if crossings is None:
    print("TODO not implemented yet — expected a short list of crossing pairs")

TODO not implemented yet — expected a short list of crossing pairs


### Task 4 · (4 min) Record the decision

Write the two times on the benchmark sheet, then state which view you would ship, for which audience, and why.

In [15]:
# TODO: record both times and write your one-sentence shipping decision.
bench = None
if bench is None:
    print("TODO not implemented yet — record your two times and the audience each view serves")

TODO not implemented yet — record your two times and the audience each view serves
